### Import dependencies

In [5]:
import os

from dotenv import load_dotenv
from dvc.database import client
from google import genai

from tqdm.autonotebook import tqdm as notebook_tqdm
import pynvml

### Load the api_key

In [32]:
load_dotenv(dotenv_path='../.env')


api_key = os.getenv("GOOGLE_API_KEY")
DEFAULT_GEMINI_MODEL = "gemini-2.5-flash"
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

gemini_client = genai.Client(api_key=api_key)

# ---- testing purpose ----
# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents="Hello gemini!"
# )
# print(response.text)

def ask_gemini (pmpt , **kwargs):
    return gemini_client.models.generate_content(
    model=DEFAULT_GEMINI_MODEL,
    contents=pmpt
)

In [33]:
## testing purpose
response = ask_gemini("Explain HR leave policy in simple terms in one line ")
print(response.text)

It's the company's rules for when and how employees can take approved time off from work for reasons like sickness, vacation, or personal needs.


### Load the documents

In [34]:
from langchain_community.document_loaders import PyPDFLoader

file_paths = [
    "../data/raw/hr_policy.pdf",
    "../data/raw/complaints_policy.pdf",
    "../data/raw/leave_policy.pdf"
]

documents = []
for path in file_paths:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents")


Loaded 3 documents


### Chunking Experiment

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")

Split 3 documents into 25 chunks


### Embedding Experiment

In [38]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c.page_content for c in chunks]

embeddings = embedding_model.encode(texts)

print(len(embeddings))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10907.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


25


### Vector Database

In [43]:
import chromadb

client = chromadb.Client()

try:
    client.delete_collection("docs")
except Exception as e:
    print("have issue cannot delete the collection name.",e)

collection = client.create_collection("docs")

texts = [c.page_content for c in chunks]
metadatas = [c.metadata for c in chunks]


collection.add(
    documents=texts,
    metadatas=metadatas,
    ids=[str(i) for i in range(len(texts))]
)
print(f"Added {len(texts)} chunks with metadata to ChromaDB.")

Added 25 chunks with metadata to ChromaDB.


In [44]:
metadatas

[{'producer': 'ReportLab PDF Library - www.reportlab.com',
  'creator': '(unspecified)',
  'creationdate': '2026-03-22T05:09:19+00:00',
  'author': '(anonymous)',
  'keywords': '',
  'moddate': '2026-03-22T05:09:19+00:00',
  'subject': '(unspecified)',
  'title': '(anonymous)',
  'trapped': '/False',
  'source': '../data/raw/hr_policy.pdf',
  'total_pages': 1,
  'page': 0,
  'page_label': '1'},
 {'producer': 'ReportLab PDF Library - www.reportlab.com',
  'creator': '(unspecified)',
  'creationdate': '2026-03-22T05:09:19+00:00',
  'author': '(anonymous)',
  'keywords': '',
  'moddate': '2026-03-22T05:09:19+00:00',
  'subject': '(unspecified)',
  'title': '(anonymous)',
  'trapped': '/False',
  'source': '../data/raw/hr_policy.pdf',
  'total_pages': 1,
  'page': 0,
  'page_label': '1'},
 {'producer': 'ReportLab PDF Library - www.reportlab.com',
  'creator': '(unspecified)',
  'creationdate': '2026-03-22T05:09:19+00:00',
  'author': '(anonymous)',
  'keywords': '',
  'moddate': '2026-03-2

### Tool Creation (LangChain)

In [40]:
def search_policy(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    formatted_context = ""

    for i in range(len(docs)):
        source_path = metas[i].get('source', 'Unknown Document')
        filename = os.path.basename(source_path)


        page_num = metas[i].get('page', -1) + 1

        formatted_context += f"--- SOURCE: {filename} (Page {page_num}) ---\n{docs[i]}\n\n"

    return formatted_context

### HR Agent

In [41]:
def hr_agent(question):
    context = search_policy(question)

    prmpt = f"""
    You are an enterprise HR assistant.

    Context:
    {context}

    Question:
    {question}

    Instructions:
    1. Answer the question professionally based ONLY on the provided context.
    2. You MUST cite your sources in your answer.
    3. Use the format [Filename, Page X] at the end of sentences or paragraphs where you use that information.
    """

    response = ask_gemini(prmpt)

    return response.text


## Testing
print(hr_agent("What is maternity leave policy?"))

The company offers 12 weeks of fully paid maternity leave [leave_policy.pdf, Page 1]. This leave is provided for the birth or legal adoption of a child [leave_policy.pdf, Page 1]. To be eligible for maternity leave, employees must have completed at least 12 months of continuous service [leave_policy.pdf, Page 1].


In [45]:
print(hr_agent("What is Investigation Protocol policy?"))

The Investigation Protocol policy outlines that upon receipt of a complaint, HR will launch an impartial investigation within 48 hours [complaints_policy.pdf, Page 1]. The investigator is responsible for interviewing the complainant, the accused, and any relevant witnesses [complaints_policy.pdf, Page 1]. During this process, all parties are expected to maintain strict confidentiality [complaints_policy.pdf, Page 1].
